# 01a_keithley_acq_waveform_inter_terminal


# Preamble

## Preamble for Jupyter

In [ ]:
if 0:  # if True, display full output in Jupyter, not only last result
    from IPython.core.interactiveshell import InteractiveShell

    InteractiveShell.ast_node_interactivity = "all"  # type: ignore

from IPython.display import (
    display,  # noqa: F401
    Math,  # noqa: F401
)  # to display results in Jupyter, and to print in Latex format (not only for sympy)

## Example display Latex
# display(Math('\\pi =' + f'{np.pi:.6f}'))
# display(Math('\\sin(\\theta)'))


In [ ]:
%load_ext autoreload
%autoreload 2

## Preamble Main Libraries and a few usefull constants

In [ ]:
import sys  # noqa: F401
import os  # noqa: F401

import numpy as np
import time
from datetime import datetime
import itertools

import pandas as pd
import plotly.graph_objects as go

# sys.path.append(os.environ['USERPROFILE'] + '\\workspace\\python\\libs\\wgTools')
# import wgutils as wgu
from scipy import constants

deg2rad = np.deg2rad(1)
rad2deg = np.rad2deg(1)
eps = np.finfo(np.float64).eps
hc = constants.value("inverse meter-electron volt relationship")

In [ ]:
import keithley_utils as kthu  # your serial helper module;

## Preamble Plotly

In [ ]:
import plotly.colors

colors_plotly = plotly.colors.DEFAULT_PLOTLY_COLORS


def custom_layout(xlabel=None, ylabel=None, title=None, template="seaborn"):
    return go.Layout(
        template=template,  # Set the template
        width=800,  # Set the figure width
        height=600,  # Set the figure height
        font=dict(family="Georgia", size=14, color="#3B3B3B"),  # General font settings
        title=dict(
            text=title,
            font=dict(size=24),
            x=0.5,
            xanchor="center",
        ),
        xaxis_title=dict(text=xlabel, font=dict(size=18)),
        yaxis_title=dict(text=ylabel, font=dict(size=18)),
        xaxis=dict(tickfont=dict(size=16)),  # Set X-axis tick font size
        yaxis=dict(tickfont=dict(size=16)),  # Set Y-axis tick font size
    )


# initialize figures with something like
# plotly_fig = go.Figure(layout=custom_layout("Frequency [Hz]", "Amplitude [Volts]", fname))


## Local Functions

In [ ]:
def datenow_str():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

parse `TRAC:DATA?` response formatted as `READ,TIME` into arrays

In [ ]:
def parse_trac_data(raw):
    """
    Parse a comma-separated stream of READ,TIME,READ,TIME,... into lists.
    Returns (reads: list[float], times: list[float_or_str]).
    """
    parts = [p.strip() for p in raw.strip().split(",") if p.strip() != ""]
    reads, times = [], []
    for i in range(0, len(parts), 2):
        reads.append(float(parts[i]))
    for i in range(0, len(parts), 2):
        times.append(float(parts[i + 1]))
    df = pd.DataFrame({"Current_Amps": reads, "Time_Secs": times})
    return df

## Local Constants

In [ ]:
POWER_LINE_FREQ = 60.0  # set to 50.0 if on 50 Hz mains
POWER_LINE_PERIOD = 1 / POWER_LINE_FREQ

## Local Variables

In [ ]:
buffer_df = {}  # to store results from multiple acquisitions if desired
_run_counter = itertools.count()  # to keep track of acquisition runs if doing multiple acquisitions in a loop

In [ ]:
import itertools

In [ ]:
save_plot = False
close_connections_at_end = False
DEBUG = False

# Main

## Detect Keithley Devices

In [ ]:
devs = kthu.detect_keithley_devices(baudrate=None, verbose=True)

### Select serial port

In [ ]:
SERIALPORT = devs[0]["port"]  # type: ignore

if SERIALPORT is None:
    kthu.print_verbose("[ERROR] No valid serial port found for device.", color="red", bold=True)
else:
    kthu.print_verbose(f"[INFO] Using serial port: {SERIALPORT}", color="blue", bold=True)

## Reset Instrument and Check for Errors

In [ ]:
_ = kthu.reset_instrument(SERIALPORT, verbose=True)

## Setup Acquisition Parameters

In [ ]:
1 / POWER_LINE_PERIOD

In [ ]:
NPLC_SP = 0.5
NUM_POINTS = 120
CURR_RANGE = None  # None for autorange, or set to a specific range in amps (e.g. 1e-6 for 1 uA range)
CURR_RANGE = 1e-9

cal_acq_time = NUM_POINTS * NPLC_SP * POWER_LINE_PERIOD
print(f"[INFO] Calculated acquisition time: {cal_acq_time:.3f} seconds")

calc_integration_time = NPLC_SP * POWER_LINE_PERIOD
print(f"[INFO] Calculated integration time: {calc_integration_time * 1e3:.3f} ms\n\n\n")


actual_range, actual_nplc = kthu.set_range(SERIALPORT, set_curr_range=CURR_RANGE, nplc=NPLC_SP, verbose=True)


In [ ]:
actual_acq_time = NUM_POINTS * actual_nplc * POWER_LINE_PERIOD
actual_integration_time = actual_nplc * POWER_LINE_PERIOD

print("\n[INFO] Starting waveform acquisition. Set Points:")
print(f"\tNPLC SP:\t\t\t{NPLC_SP:.3f}")
print(f"\tNPLC Actual:\t\t\t{actual_nplc:.3f}")
print(f"\tRange SP:\t\t\t{CURR_RANGE:.6g} A") if CURR_RANGE is not None else print("\tRange SP:\t\t\tAUTO RANGE")
print(f"\tRange Actual:\t\t\t{actual_range:.6g} A")
print(f"\tNUM_POINTS:\t\t\t{NUM_POINTS} points")
print(f"\tCalc Acq Time SP:\t\t{cal_acq_time:.3f} s")
print(f"\tActual Acq Time:\t\t{actual_acq_time:.3f} s")
print(f"\tIntegration Time SP:\t\t{calc_integration_time * 1e3:.3f} ms")
print(f"\tIntegration Time Actual:\t{actual_integration_time * 1e3:.3f} ms")

## Zero Instrument and Check for Errors

In [ ]:
# %% Zero Instrument and Check for Errors
zero_val = kthu.zero_instrument(SERIALPORT, verbose=True)


## Setup Acquisition

In [ ]:
# %% Setup Acquisition
kthu.setup_waveform_acquisition(
    SERIALPORT,
    num_points=NUM_POINTS,
    verbose=True,
    debug=False,
)


## Acquire Waveform, parse data, and load to buffer variable

In [ ]:
_total_t_start = time.time()
raw = kthu.acq_waveform(SERIALPORT, verbose=True)
kthu.print_verbose(
    f"[INFO] Acquisition Finished. Elapsed time: {time.time() - _total_t_start:.2f} s", color="purple", bold=True
)

kthu.print_verbose("\n[INFO] Parsing Data to a DataFrame...", color="purple")

df_acq = parse_trac_data(raw)
df_acq["NPLC"] = NPLC_SP
df_acq["Range"] = actual_range
df_acq["Time_msecs"] = df_acq["Time_Secs"] * 1000
kthu.print_verbose(f"[RESULTS] Acquired samples: {len(df_acq)}", color="green")
kthu.print_verbose("[INFO] Parsing Data DONE...", color="purple")


kthu.print_verbose("\n[INFO] Loading DataFrame to a Buffer dictionary...", color="purple")
buffer_df[next(_run_counter)] = df_acq  # store in buffer if doing multiple acquisitions in a loop
kthu.print_verbose(f"[INFO] DONE! Buffer contain {len(buffer_df)} acquisition(s).", color="purple")


### Review Results

In [ ]:
kthu.print_verbose("[RESULTS] Samples DataFrame info:", color="green", bold=True)
print(df_acq.info())
kthu.print_verbose("[RESULTS] Samples DataFrame describe:", color="green", bold=True)
print(df_acq.describe())


# Post Processing

## Load last Acquisition for Post Processing

In [ ]:
run_ids = list(buffer_df.keys())
run_ids

In [ ]:
df = buffer_df[run_ids[-1]]  # Load last Acquisition for Post Processing

In [ ]:
print("\n[POSTPROCESSING] Post Processing STARTED")

print("\n[POSTPROCESSING] Plotting data with Plotly...")
fig = go.Figure()

_col_x = "Time_Secs"
_col_y = "Current_Amps"

fig.add_trace(go.Scatter(x=df[_col_x], y=df[_col_y], mode="lines+markers", name=_col_y))

fig.update_layout(
    title="LabJack Data Acquisition",
    xaxis_title="Time (s)",
    yaxis_title="Current (A)",
    hovermode="x unified",
    template="plotly",
)
# Save HTML and open in VS Code webview
html_file = f"Results\\{datenow_str()}_labjack_acquisition_plot.html"

if save_plot:
    fig.write_html(html_file)
    print(f"[POSTPROCESSING] Plot saved to .\\{html_file}\n")

fig.show()

In [ ]:
# Force close serial connection at the end if desired
import serial

if False:
    serial.Serial(SERIALPORT).close()
    kthu.print_verbose(f"[INFO] Serial connection on {SERIALPORT} closed.", color="blue", bold=True)
else:
    kthu.print_verbose(f"[INFO] Serial connection on {SERIALPORT} left open.", color="cyan", bold=True)